# 📁 Часть 1: Работа со статическими данными

> Данные загружаются из Google Drive (CSV-файлы).
> 
> - Визиты: [https://drive.google.com/uc?export=download&id=1QosQQ4RRNR9rkL4t7sB707h2Uy0XfYJe]
> - Регистрации: [https://drive.google.com/uc?export=download&id=1AeQz0kaSgz0lxYSDtuNm36muhy5fRCzZ]

In [1]:
import pandas as pd
import requests
import warnings
warnings.filterwarnings('ignore')  # Скрыть предупреждение о небезопасном соединении

VISITS_URL = "https://drive.google.com/uc?export=download&id=1QosQQ4RRNR9rkL4t7sB707h2Uy0XfYJe"
REG_URL = "https://drive.google.com/uc?export=download&id=1AeQz0kaSgz0lxYSDtuNm36muhy5fRCzZ"

print("📥 Скачиваем данные...")

r_visits = requests.get(VISITS_URL, verify=False)
r_reg = requests.get(REG_URL, verify=False)

r_visits.raise_for_status()
r_reg.raise_for_status()

with open("visits.csv", "wb") as f:
    f.write(r_visits.content)
with open("registrations.csv", "wb") as f:
    f.write(r_reg.content)

print("✅ Файлы сохранены!")

df_visits = pd.read_csv("visits.csv")
df_reg = pd.read_csv("registrations.csv")

print(f"\n📊 Визиты: {df_visits.shape}")
print(f"📊 Регистрации: {df_reg.shape}")
display(df_visits.head())
display(df_reg.head())

📥 Скачиваем данные...
✅ Файлы сохранены!

📊 Визиты: (1000, 4)
📊 Регистрации: (1000, 5)


,uuid,platform,user_agent,date
0,1de9ea66-70d3-4a1f-8735-df5ef7697fb9,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-03-01T13:29:22
1,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-01T16:44:28
2,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-06T06:12:36
3,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-01T20:16:37
4,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-05T17:42:47


,date,user_id,email,platform,registration_type
0,2023-03-01T00:25:39,8838849,joseph95@example.org,web,google
1,2023-03-01T14:53:01,8741065,janetsuarez@example.net,web,yandex
2,2023-03-01T14:27:36,1866654,robert67@example.org,web,google
3,2023-03-01T02:42:34,1577584,elam@example.net,web,apple
4,2023-03-01T10:27:14,4765395,stephanie68@example.net,web,yandex


In [2]:
import pandas as pd
import requests
import os
import warnings
warnings.filterwarnings('ignore')

print("🔄 Проверка и загрузка всех необходимых данных...")

API_BASE = "https://data-charts-api.hexlet.app"
START, END = "2023-03-01", "2023-09-01"

# ────────────────────────────────────────────────────
# 1. Визиты и Регистрации (API или локальный CSV)
# ─────────────────────────────────────────────────────
def load_api_dataset(name, filename, is_visits=True):
    var_name = 'df_visits_api' if is_visits else 'df_reg_api'
    if var_name in globals() and not globals()[var_name].empty:
        print(f"  ✅ {name}: уже в памяти")
        return globals()[var_name]
    if os.path.exists(filename):
        print(f"  📂 {name}: загружаю из {filename}")
        df = pd.read_csv(filename)
        globals()[var_name] = df
        return df
    
    print(f"  🌐 {name}: скачиваю из API...")
    endpoint = "visits" if is_visits else "registrations"
    url = f"{API_BASE}/{endpoint}?begin={START}&end={END}"
    r = requests.get(url, verify=False, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df.to_csv(filename, index=False)
    globals()[var_name] = df
    print(f"  ✅ {name}: сохранено и загружено ({len(df)} строк)")
    return df

if 'df_visits_api' not in globals():
    df_visits_api = load_api_dataset("Визиты", "visits_api.csv", is_visits=True)
if 'df_reg_api' not in globals():
    df_reg_api = load_api_dataset("Регистрации", "registrations_api.csv", is_visits=False)

# ─────────────────────────────────────────────────────
# 2. Рекламные кампании
# ─────────────────────────────────────────────────────
if 'df_ads' not in globals():
    if os.path.exists('ads.csv'):
        print("  📂 Реклама: загружаю из ads.csv")
        df_ads = pd.read_csv('ads.csv')
    else:
        print("  🌐 Реклама: скачиваю с Google Drive...")
        url = "https://drive.google.com/uc?export=download&id=12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj"
        r = requests.get(url, verify=False, timeout=30)
        r.raise_for_status()
        with open('ads.csv', 'wb') as f: f.write(r.content)
        df_ads = pd.read_csv('ads.csv')
    globals()['df_ads'] = df_ads
    print(f"  ✅ Реклама: загружено ({len(df_ads)} строк)")

# ─────────────────────────────────────────────────────
# 3. Финальная проверка
# ─────────────────────────────────────────────────────
required = ['df_visits_api', 'df_reg_api', 'df_ads']
missing = [v for v in required if v not in globals() or globals()[v].empty]

if missing:
    print(f"\n❌ ОШИБКА: Не удалось загрузить {missing}")
    print(" Проверьте интернет или наличие файлов в папке проекта")
else:
    print("\n ВСЕ ДАННЫЕ ГОТОВЫ!")
    print(f"    Визиты: {len(df_visits_api)} | 📝 Регистрации: {len(df_reg_api)} | 📢 Реклама: {len(df_ads)}")

🔄 Проверка и загрузка всех необходимых данных...
  📂 Визиты: загружаю из visits_api.csv
  📂 Регистрации: загружаю из registrations_api.csv
  📂 Реклама: загружаю из ads.csv
  ✅ Реклама: загружено (159 строк)

 ВСЕ ДАННЫЕ ГОТОВЫ!
    Визиты: 263459 | 📝 Регистрации: 21836 | 📢 Реклама: 159


# 🌐 Часть 2: Запросы к API

> Данные получаются в реальном времени через HTTP-запросы.
> 
> *Период:* `2023-03-01` — `2023-09-01`  
> *API Base URL:* `https://data-charts-api.hexlet.app`

In [3]:
import pandas as pd
import requests
import warnings
warnings.filterwarnings('ignore')  # Скрыть предупреждение о небезопасном соединении

# 📅 Параметры периода
START_DATE = "2023-03-01"
END_DATE = "2023-09-01"

# 🔗 Базовый URL API
BASE_URL = "https://data-charts-api.hexlet.app"

print(f"📡 Запрашиваем данные за период: {START_DATE} — {END_DATE}")
print("-" * 60)

# ⚠️ Параметр для обхода SSL-ошибки
SSL_VERIFY = False

# ─────────────────────────────────────────────────────
# 1. Запрос данных по визитам
# ─────────────────────────────────────────────────────
visits_url = f"{BASE_URL}/visits?begin={START_DATE}&end={END_DATE}"
print(f"🔹 Запрос визитов: {visits_url}")

try:
    response_visits = requests.get(visits_url, timeout=30, verify=SSL_VERIFY)
    response_visits.raise_for_status()
    visits_data = response_visits.json()
    df_visits_api = pd.DataFrame(visits_data)
    print(f"✅ Визиты: получено {len(df_visits_api)} записей")
except requests.exceptions.RequestException as e:
    print(f"❌ Ошибка при запросе визитов: {e}")
    df_visits_api = pd.DataFrame()

# ─────────────────────────────────────────────────────
# 2. Запрос данных по регистрациям
# ─────────────────────────────────────────────────────
reg_url = f"{BASE_URL}/registrations?begin={START_DATE}&end={END_DATE}"
print(f"🔹 Запрос регистраций: {reg_url}")

try:
    response_reg = requests.get(reg_url, timeout=30, verify=SSL_VERIFY)
    response_reg.raise_for_status()
    reg_data = response_reg.json()
    df_reg_api = pd.DataFrame(reg_data)
    print(f"✅ Регистрации: получено {len(df_reg_api)} записей")
except requests.exceptions.RequestException as e:
    print(f"❌ Ошибка при запросе регистраций: {e}")
    df_reg_api = pd.DataFrame()

# ─────────────────────────────────────────────────────
# 3. Быстрая проверка данных
# ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📋 ПРЕДВАРИТЕЛЬНЫЙ ПРОСМОТР")
print("=" * 60)

if not df_visits_api.empty:
    print("\n🔹 ВИЗИТЫ (первые 3 строки):")
    display(df_visits_api.head(3))
    print(f"\nКолонки: {list(df_visits_api.columns)}")
    
if not df_reg_api.empty:
    print("\n🔹 РЕГИСТРАЦИИ (первые 3 строки):")
    display(df_reg_api.head(3))
    print(f"\nКолонки: {list(df_reg_api.columns)}")

print("\n✅ Запросы к API завершены!")

📡 Запрашиваем данные за период: 2023-03-01 — 2023-09-01
------------------------------------------------------------
🔹 Запрос визитов: https://data-charts-api.hexlet.app/visits?begin=2023-03-01&end=2023-09-01
✅ Визиты: получено 263459 записей
🔹 Запрос регистраций: https://data-charts-api.hexlet.app/registrations?begin=2023-03-01&end=2023-09-01
✅ Регистрации: получено 21836 записей

📋 ПРЕДВАРИТЕЛЬНЫЙ ПРОСМОТР

🔹 ВИЗИТЫ (первые 3 строки):


,visit_id,platform,user_agent,datetime
0,a88b7acc-5044-4a76-9b33-648bc02441fe,android,Mozilla/5.0 (Linux; Android 13; SAMSUNG SM-A54...,2023-04-17T11:56:33
1,30c72ea4-d80b-4ec6-81bf-8677832848f2,android,Mozilla/5.0 (Linux; Android 13; SAMSUNG SM-A54...,2023-04-17T21:52:00
2,1a166b9b-84ba-43f1-b49f-5097d724dd4f,android,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,2023-04-17T01:33:43



Колонки: ['visit_id', 'platform', 'user_agent', 'datetime']

🔹 РЕГИСТРАЦИИ (первые 3 строки):


,datetime,user_id,email,platform,registration_type
0,2023-03-01T07:40:13,2e0f6bb8-b029-4f45-a786-2b53990d37f1,ebyrd@example.org,web,google
1,2023-03-01T13:14:00,f007f97c-9d8b-48b5-af08-119bb8f6d9b6,knightgerald@example.org,web,email
2,2023-03-01T03:05:50,24ff46ae-32b3-4a74-8f27-7cf0b8f32f15,cherylthompson@example.com,web,apple



Колонки: ['datetime', 'user_id', 'email', 'platform', 'registration_type']

✅ Запросы к API завершены!


# 🌐 Часть 3: Расчет метрик

In [4]:
import pandas as pd

# 🔹 0. Преобразуем строки в даты (ОБЯЗАТЕЛЬНЫЙ ШАГ!)
df_visits_api['datetime'] = pd.to_datetime(df_visits_api['datetime'])
df_reg_api['datetime'] = pd.to_datetime(df_reg_api['datetime'])

# 🔹 1. Подготовка визитов
df_v = df_visits_api.copy()

# Исключаем ботов (слово 'bot' в user_agent, без учета регистра)
df_v = df_v[~df_v['user_agent'].str.contains('bot', case=False, na=False)]

# Оставляем только последний визит для каждого visit_id
df_v = df_v.sort_values('datetime').drop_duplicates(subset=['visit_id'], keep='last')

# Выделяем дату для группировки (теперь .dt работает!)
df_v['date_group'] = df_v['datetime'].dt.date

# Агрегируем: считаем визиты по дате и платформе
visits_agg = df_v.groupby(['date_group', 'platform']).size().reset_index(name='visits')

# 🔹 2. Подготовка регистраций
df_r = df_reg_api.copy()
df_r['date_group'] = df_r['datetime'].dt.date

# Агрегируем регистрации по дате и платформе
reg_agg = df_r.groupby(['date_group', 'platform']).size().reset_index(name='registrations')

# 🔹 3. Объединение таблиц
df_conv = pd.merge(visits_agg, reg_agg, on=['date_group', 'platform'], how='outer').fillna(0)

df_conv['visits'] = df_conv['visits'].astype(int)
df_conv['registrations'] = df_conv['registrations'].astype(int)

# 🔹 4. Расчет конверсии (в процентах)
df_conv['conversion'] = df_conv.apply(
    lambda row: (row['registrations'] / row['visits']) * 100 if row['visits'] > 0 else 0.0,
    axis=1
).round(2)

# 🔹 5. Сортировка и порядок колонок
df_conv = df_conv.sort_values('date_group').reset_index(drop=True)
df_conv = df_conv[['date_group', 'platform', 'visits', 'registrations', 'conversion']]

# 🔹 6. Сохранение в JSON
df_conv.to_json('conversion.json')

print("✅ Метрики рассчитаны и сохранены в conversion.json")
print(f"📊 Всего строк: {len(df_conv)}")
display(df_conv.head(10))

✅ Метрики рассчитаны и сохранены в conversion.json
📊 Всего строк: 552


,date_group,platform,visits,registrations,conversion
0,2023-03-01,android,75,61,81.33
1,2023-03-01,ios,22,18,81.82
2,2023-03-01,web,279,8,2.87
3,2023-03-02,android,67,59,88.06
4,2023-03-02,ios,31,24,77.42
5,2023-03-02,web,515,23,4.47
6,2023-03-03,android,26,22,84.62
7,2023-03-03,ios,40,34,85.00
8,2023-03-03,web,617,51,8.27
9,2023-03-04,android,94,77,81.91


In [5]:
# Корректно форматируем в JSON

In [6]:
import json

# Читаем и красиво выводим первые 20 строк файла
with open('conversion.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    
# Показываем структуру
print("📄 Формат JSON:")
for key in data.keys():
    print(f"  {key}: {len(data[key])} значений")

# Пример первых 3 записей
print("\n📋 Пример данных:")
for i in range(min(3, len(data['date_group']))):
    print(f"  [{i}] date={data['date_group'][str(i)]}, "
          f"platform={data['platform'][str(i)]}, "
          f"conv={data['conversion'][str(i)]}%")

📄 Формат JSON:
  date_group: 552 значений
  platform: 552 значений
  visits: 552 значений
  registrations: 552 значений
  conversion: 552 значений

📋 Пример данных:
  [0] date=1677628800000, platform=android, conv=81.33%
  [1] date=1677628800000, platform=ios, conv=81.82%
  [2] date=1677628800000, platform=web, conv=2.87%


# Задание 4 Добавляем рекламы

In [7]:
import pandas as pd
import requests
import warnings
warnings.filterwarnings('ignore')

# 🔗 Ссылка на CSV с рекламой (преобразована для прямого скачивания)
ADS_URL = "https://drive.google.com/uc?export=download&id=12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj"

print("📥 Загружаем данные по рекламе...")

# Скачиваем файл
r_ads = requests.get(ADS_URL, verify=False)
r_ads.raise_for_status()

# Сохраняем и читаем
with open("ads.csv", "wb") as f:
    f.write(r_ads.content)

df_ads = pd.read_csv("ads.csv")
print(f"✅ Реклама: {len(df_ads)} записей")
display(df_ads.head(3))

📥 Загружаем данные по рекламе...
✅ Реклама: 159 записей


,date,utm_source,utm_medium,utm_campaign,cost
0,2023-03-01T10:54:41,google,cpc,advanced_algorithms_series,212
1,2023-03-02T10:32:35,google,cpc,advanced_algorithms_series,252
2,2023-03-03T19:21:40,google,cpc,advanced_algorithms_series,202


In [8]:
# 🔹 1. Подготовка данных по рекламе
df_ads['date'] = pd.to_datetime(df_ads['date']).dt.date

# Агрегируем рекламу по дате: суммируем cost, берём первое название кампании за день
ads_agg = df_ads.groupby('date').agg({
    'cost': 'sum',
    'utm_campaign': 'first'  # Если кампаний несколько — берём первую
}).reset_index()
ads_agg.rename(columns={'date': 'date_group'}, inplace=True)

print(f"📊 Рекламных дней: {len(ads_agg)}")

📊 Рекламных дней: 159


In [9]:
import pandas as pd
import requests
import warnings
import os
warnings.filterwarnings('ignore')

# 🔹 Функция для загрузки данных (умная: пробует разные источники)
def load_data_smart(api_url, csv_filename, endpoint_name):
    """
    Пытается загрузить данные:
    1. Из памяти (если переменная уже есть)
    2. Из локального CSV (если скачивали раньше)
    3. Из API (если ничего не найдено)
    """
    # 1. Проверяем память
    if endpoint_name == 'visits' and 'df_visits_api' in globals():
        print(f"✅ {endpoint_name}: используем данные из памяти")
        return df_visits_api
    if endpoint_name == 'reg' and 'df_reg_api' in globals():
        print(f"✅ {endpoint_name}: используем данные из памяти")
        return df_reg_api
    
    # 2. Проверяем локальный CSV
    if os.path.exists(csv_filename):
        print(f"✅ {endpoint_name}: загружаем из {csv_filename}")
        return pd.read_csv(csv_filename)
    
    # 3. Запрашиваем из API (с обходом SSL)
    print(f"⏳ {endpoint_name}: запрашиваем из API...")
    try:
        r = requests.get(api_url, timeout=60, verify=False)
        r.raise_for_status()
        df = pd.DataFrame(r.json())
        # Сохраняем локально на будущее
        df.to_csv(csv_filename, index=False)
        print(f"✅ {endpoint_name}: получено {len(df)} записей, сохранено в {csv_filename}")
        return df
    except Exception as e:
        print(f"❌ Ошибка загрузки {endpoint_name}: {e}")
        return pd.DataFrame()

# 🔹 Параметры API
START_DATE = "2023-03-01"
END_DATE = "2023-09-01"
BASE_URL = "https://data-charts-api.hexlet.app"

# Загружаем визиты и регистрации
df_visits_api = load_data_smart(
    f"{BASE_URL}/visits?begin={START_DATE}&end={END_DATE}",
    "visits_api.csv",
    "visits"
)

df_reg_api = load_data_smart(
    f"{BASE_URL}/registrations?begin={START_DATE}&end={END_DATE}",
    "registrations_api.csv", 
    "reg"
)

# Проверка
if df_visits_api.empty or df_reg_api.empty:
    print("⚠️ Не удалось загрузить данные! Проверьте интернет или выполните ячейку с API вручную.")
else:
    print(f"\n🎉 Данные готовы: визиты={len(df_visits_api)}, регистрации={len(df_reg_api)}")

✅ visits: используем данные из памяти
✅ reg: используем данные из памяти

🎉 Данные готовы: визиты=263459, регистрации=21836


In [10]:
# 🔹 Агрегация конверсий ТОЛЬКО по дате (суммируем по всем платформам)

# === ВАЖНО: преобразуем строки в даты, если они ещё не datetime ===
if df_visits_api['datetime'].dtype == 'object':
    df_visits_api['datetime'] = pd.to_datetime(df_visits_api['datetime'])
if df_reg_api['datetime'].dtype == 'object':
    df_reg_api['datetime'] = pd.to_datetime(df_reg_api['datetime'])

# Визиты: фильтруем ботов, оставляем последний визит по visit_id
df_v = df_visits_api.copy()
df_v = df_v[~df_v['user_agent'].str.contains('bot', case=False, na=False)]
df_v = df_v.sort_values('datetime').drop_duplicates(subset=['visit_id'], keep='last')
df_v['date_group'] = df_v['datetime'].dt.date  # Теперь .dt работает!
visits_by_date = df_v.groupby('date_group').size().reset_index(name='visits')

# Регистрации: просто группируем по дате
df_r = df_reg_api.copy()
df_r['date_group'] = df_r['datetime'].dt.date
reg_by_date = df_r.groupby('date_group').size().reset_index(name='registrations')

# Объединяем визиты и регистрации по дате
conv_by_date = pd.merge(visits_by_date, reg_by_date, on='date_group', how='outer').fillna(0)
conv_by_date['visits'] = conv_by_date['visits'].astype(int)
conv_by_date['registrations'] = conv_by_date['registrations'].astype(int)

print(f"📊 Дней с активностью: {len(conv_by_date)}")
print(f"📅 Диапазон дат: {conv_by_date['date_group'].min()} — {conv_by_date['date_group'].max()}")

📊 Дней с активностью: 184
📅 Диапазон дат: 2023-03-01 — 2023-08-31


In [11]:
# 🔹 Загрузка рекламы
ADS_URL = "https://drive.google.com/uc?export=download&id=12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj"

print("📥 Загружаем данные по рекламе...")
r_ads = requests.get(ADS_URL, verify=False)
r_ads.raise_for_status()

with open("ads.csv", "wb") as f:
    f.write(r_ads.content)
df_ads = pd.read_csv("ads.csv")
print(f"✅ Реклама: {len(df_ads)} записей")

# 🔹 Агрегация рекламы по дате
df_ads['date'] = pd.to_datetime(df_ads['date']).dt.date
ads_agg = df_ads.groupby('date').agg({
    'cost': 'sum',
    'utm_campaign': 'first'
}).reset_index()
ads_agg.rename(columns={'date': 'date_group'}, inplace=True)

# 🔹 Объединение с конверсиями (conv_by_date из предыдущей ячейки)
df_final = pd.merge(conv_by_date, ads_agg, on='date_group', how='left')
df_final['cost'] = df_final['cost'].fillna(0).astype(int)
df_final['utm_campaign'] = df_final['utm_campaign'].fillna('none')

# Сортировка и порядок колонок
df_final = df_final.sort_values('date_group').reset_index(drop=True)
df_final = df_final[['date_group', 'visits', 'registrations', 'cost', 'utm_campaign']]

# Сохранение в JSON
df_final.to_json('ads.json')

print("\n✅ Данные сохранены в ads.json")
print(f"📈 Всего строк: {len(df_final)}")
display(df_final.head(10))

📥 Загружаем данные по рекламе...
✅ Реклама: 159 записей

✅ Данные сохранены в ads.json
📈 Всего строк: 184


,date_group,visits,registrations,cost,utm_campaign
0,2023-03-01,376,87,212,advanced_algorithms_series
1,2023-03-02,613,106,252,advanced_algorithms_series
2,2023-03-03,683,107,202,advanced_algorithms_series
3,2023-03-04,647,159,223,advanced_algorithms_series
4,2023-03-05,707,115,265,advanced_algorithms_series
5,2023-03-06,1291,230,108,advanced_algorithms_series
6,2023-03-07,1382,124,165,advanced_algorithms_series
7,2023-03-08,1382,151,155,advanced_algorithms_series
8,2023-03-09,1064,209,124,advanced_algorithms_series
9,2023-03-10,812,112,276,advanced_algorithms_series


# Задание 5 Визуализация

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Настройка стилей
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

# Создаём папку для графиков
os.makedirs("charts", exist_ok=True)
print("✅ Папка ./charts создана")

✅ Папка ./charts создана


In [13]:
import pandas as pd
import requests
import warnings
import os
warnings.filterwarnings('ignore')

# 🔹 Умная загрузка рекламы: проверяем память → CSV → API/Google Drive
if 'df_ads' not in globals():
    print("🔍 df_ads не найден в памяти, пробуем загрузить...")
    
    # 1. Пробуем загрузить из локального CSV (если скачивали раньше)
    if os.path.exists("ads.csv"):
        print("✅ Загружаем ads.csv из локальной папки...")
        df_ads = pd.read_csv("ads.csv")
    
    # 2. Если нет CSV — скачиваем с Google Drive
    else:
        print("⏳ Скачиваем ads.csv с Google Drive...")
        ADS_URL = "https://drive.google.com/uc?export=download&id=12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj"
        try:
            r = requests.get(ADS_URL, verify=False, timeout=30)
            r.raise_for_status()
            with open("ads.csv", "wb") as f:
                f.write(r.content)
            df_ads = pd.read_csv("ads.csv")
            print("✅ Файл скачан и загружен!")
        except Exception as e:
            print(f"❌ Ошибка загрузки рекламы: {e}")
            df_ads = pd.DataFrame()  # Пустой, чтобы код не упал
else:
    print("✅ df_ads уже есть в памяти")

# Проверка
if not df_ads.empty:
    print(f"📊 Реклама: {len(df_ads)} записей, колонки: {list(df_ads.columns)}")
    display(df_ads.head(2))
else:
    print("⚠️ Внимание: df_ads пустой! Проверьте интернет или наличие файла ads.csv")

✅ df_ads уже есть в памяти
📊 Реклама: 159 записей, колонки: ['date', 'utm_source', 'utm_medium', 'utm_campaign', 'cost']


,date,utm_source,utm_medium,utm_campaign,cost
0,2023-03-01,google,cpc,advanced_algorithms_series,212
1,2023-03-02,google,cpc,advanced_algorithms_series,252


In [14]:
# === Визиты: фильтруем ботов, оставляем последний визит по visit_id ===
df_v = df_visits_api.copy()
df_v['datetime'] = pd.to_datetime(df_v['datetime'])
df_v = df_v[~df_v['user_agent'].str.contains('bot', case=False, na=False)]
df_v = df_v.sort_values('datetime').drop_duplicates(subset=['visit_id'], keep='last')
df_v['date'] = df_v['datetime'].dt.date

# === Регистрации ===
df_r = df_reg_api.copy()
df_r['datetime'] = pd.to_datetime(df_r['datetime'])
df_r['date'] = df_r['datetime'].dt.date

# === Агрегация визитов по дате и платформе ===
visits_by_date_platform = df_v.groupby(['date', 'platform']).size().unstack(fill_value=0).reset_index()
visits_by_date = df_v.groupby('date').size().reset_index(name='visits')

# === Агрегация регистраций по дате и платформе ===
reg_by_date_platform = df_r.groupby(['date', 'platform']).size().unstack(fill_value=0).reset_index()
reg_by_date = df_r.groupby('date').size().reset_index(name='registrations')

# === Конверсия по платформам ===
conv_by_platform = pd.merge(
    df_v.groupby('platform').size().reset_index(name='visits'),
    df_r.groupby('platform').size().reset_index(name='registrations'),
    on='platform', how='outer'
).fillna(0)
conv_by_platform['visits'] = conv_by_platform['visits'].astype(int)
conv_by_platform['registrations'] = conv_by_platform['registrations'].astype(int)
conv_by_platform['conversion'] = (conv_by_platform['registrations'] / conv_by_platform['visits'] * 100).round(2)

# === Реклама: подготовка для выделения ===
df_ads['date'] = pd.to_datetime(df_ads['date']).dt.date
ads_by_date = df_ads.groupby('date')['utm_campaign'].first().reset_index()

print("✅ Данные подготовлены")

✅ Данные подготовлены


Построение и сохранение графиков

In [25]:
# === Итоговые визиты ===
plt.figure(figsize=(14, 6), dpi=100)

# Столбчатая диаграмма
bars = plt.bar(visits_by_date['date'], visits_by_date['visits'], 
               color='skyblue', edgecolor='black', linewidth=0.5)

# Подписи значений над столбцами
for bar, val in zip(bars, visits_by_date['visits']):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 15, 
             f'{int(val)}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Оформление
plt.title('Total Visits', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('date_group', fontsize=11)
plt.ylabel('visits', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()

# Сохранение
plt.savefig('charts/total_visits.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/total_visits.png обновлён")

✅ charts/total_visits.png обновлён


In [26]:
# === Визиты с разбивкой по платформам ===
plt.figure(figsize=(14, 6), dpi=100)

# Данные уже подготовлены в visits_by_date_platform через unstack
# Убедимся, что колонки в нужном порядке
platforms = ['android', 'ios', 'web']
colors = {'android': '#4A90E2', 'ios': '#F5A623', 'web': '#50E3C2'}

# Рисуем нижний слой (android)
p1 = plt.bar(visits_by_date_platform['date'], visits_by_date_platform.get('android', 0), 
             color=colors['android'], label='android', edgecolor='white', linewidth=0.5)

# Рисуем средний слой (ios) - поверх android
p2 = plt.bar(visits_by_date_platform['date'], visits_by_date_platform.get('ios', 0), 
             bottom=visits_by_date_platform.get('android', 0),
             color=colors['ios'], label='ios', edgecolor='white', linewidth=0.5)

# Рисуем верхний слой (web) - поверх ios
p3 = plt.bar(visits_by_date_platform['date'], visits_by_date_platform.get('web', 0), 
             bottom=visits_by_date_platform.get('android', 0) + visits_by_date_platform.get('ios', 0),
             color=colors['web'], label='web', edgecolor='white', linewidth=0.5)

# Оформление
plt.title('Visits by Platform (Stacked)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('date_group', fontsize=11)
plt.ylabel('visits', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(title='platform', loc='upper right')
plt.tight_layout()

# Сохранение
plt.savefig('charts/visits_by_platform.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/visits_by_platform.png обновлён")

✅ charts/visits_by_platform.png обновлён


In [27]:
# === Итоговые регистрации ===
plt.figure(figsize=(14, 6), dpi=100)

# Столбчатая диаграмма (тот же светло-голубой цвет)
bars = plt.bar(reg_by_date['date'], reg_by_date['registrations'], 
               color='skyblue', edgecolor='black', linewidth=0.5)

# Подписи значений над столбцами
for bar, val in zip(bars, reg_by_date['registrations']):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5, 
             f'{int(val)}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Оформление
plt.title('Total Registrations', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('date_group', fontsize=11)
plt.ylabel('registrations', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()

# Сохранение
plt.savefig('charts/total_registrations.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/total_registrations.png обновлён")

✅ charts/total_registrations.png обновлён


In [28]:
# === Регистрации с разбивкой по платформам ===
plt.figure(figsize=(14, 6), dpi=100)

# Данные уже подготовлены в reg_by_date_platform через unstack
# Убедимся, что колонки в нужном порядке для стека
platforms = ['android', 'ios', 'web']
colors = {'android': '#4A90E2', 'ios': '#F5A623', 'web': '#50E3C2'}

# Нижний слой: android
p1 = plt.bar(reg_by_date_platform['date'], reg_by_date_platform.get('android', 0), 
             color=colors['android'], label='android', edgecolor='white', linewidth=0.5)

# Средний слой: ios (кладём поверх android)
p2 = plt.bar(reg_by_date_platform['date'], reg_by_date_platform.get('ios', 0), 
             bottom=reg_by_date_platform.get('android', 0),
             color=colors['ios'], label='ios', edgecolor='white', linewidth=0.5)

# Верхний слой: web (кладём поверх ios)
p3 = plt.bar(reg_by_date_platform['date'], reg_by_date_platform.get('web', 0), 
             bottom=reg_by_date_platform.get('android', 0) + reg_by_date_platform.get('ios', 0),
             color=colors['web'], label='web', edgecolor='white', linewidth=0.5)

# Оформление
plt.title('Registrations by Platform (Stacked)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('date_group', fontsize=11)
plt.ylabel('registrations', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(title='platform', loc='upper right')
plt.tight_layout()

# Сохранение
plt.savefig('charts/registrations_by_platform.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/registrations_by_platform.png обновлён")

✅ charts/registrations_by_platform.png обновлён


In [29]:
# === Конверсия по каждой платформе ===
# 1. Подготовка данных: конверсия по дням и платформам
# Группируем визиты по дате и платформе
visits_plat = df_v.groupby(['date', 'platform']).size().reset_index(name='visits')
# Группируем регистрации по дате и платформе
regs_plat = df_r.groupby(['date', 'platform']).size().reset_index(name='registrations')

# Объединяем и считаем конверсию
df_conv_plat = pd.merge(visits_plat, regs_plat, on=['date', 'platform'], how='outer').fillna(0)
df_conv_plat['visits'] = df_conv_plat['visits'].astype(int)
df_conv_plat['registrations'] = df_conv_plat['registrations'].astype(int)
df_conv_plat['conversion'] = (df_conv_plat['registrations'] / df_conv_plat['visits'] * 100).round(0)

# 2. Построение 3-х графиков друг под другом
fig, axes = plt.subplots(3, 1, figsize=(12, 14), dpi=100, sharex=True)

platforms = ['android', 'ios', 'web']
colors = {'android': '#4A90E2', 'ios': '#F5A623', 'web': '#50E3C2'}

for ax, platform in zip(axes, platforms):
    # Фильтруем данные для текущей платформы
    data = df_conv_plat[df_conv_plat['platform'] == platform].sort_values('date')
    
    if data.empty:
        continue
        
    # Рисуем линию с маркерами
    ax.plot(data['date'], data['conversion'], 
            color=colors[platform], linewidth=1.5, marker='o', markersize=4, label=platform)
    
    # Подписи с процентами
    for i, row in data.iterrows():
        ax.text(row['date'], row['conversion'] + 2, f"{int(row['conversion'])}%", 
                ha='center', va='bottom', fontsize=8, color='black')
    
    # Оформление каждого подграфика
    ax.set_title(f'Conversion {platform}', fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel('Conversion (%)', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_ylim(0, data['conversion'].max() * 1.2 + 10) # Немного места сверху для подписей
    ax.legend(loc='upper left')
    
    # Поворот дат на нижнем графике
    if platform == 'web':
        ax.tick_params(axis='x', rotation=45)

plt.xlabel('Date', fontsize=11)
plt.tight_layout()
plt.savefig('charts/conversion_by_platform.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/conversion_by_platform.png обновлён")

✅ charts/conversion_by_platform.png обновлён


In [30]:
# === Средняя конверсия ===
# Подготовка данных: конверсия по дням
df_conv_trend = pd.merge(visits_by_date, reg_by_date, on='date')
df_conv_trend['conversion'] = (df_conv_trend['registrations'] / df_conv_trend['visits'] * 100).round(0)

plt.figure(figsize=(12, 6), dpi=100)

# Линия с маркерами
plt.plot(df_conv_trend['date'], df_conv_trend['conversion'], 
         color='#4A90E2', linewidth=1.5, marker='o', markersize=6, 
         label='Общая конверсия')

# Подписи процентов (чередование над/под точкой)
for i, row in df_conv_trend.iterrows():
    offset = 1.5 if i % 2 == 0 else -2
    plt.text(row['date'], row['conversion'] + offset, f"{int(row['conversion'])}%", 
             ha='center', va='bottom' if offset > 0 else 'top', 
             fontsize=9, color='black')

# Оформление
plt.title('Average Conversion', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=11)
plt.ylabel('Conversion (%)', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.grid(True, linestyle='-', alpha=0.4)
plt.legend(loc='upper right')
plt.tight_layout()

# Сохранение как "Средняя конверсия"
plt.savefig('charts/average_conversion.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/average_conversion.png обновлён")

✅ charts/average_conversion.png обновлён


In [31]:
# === Стоимость рекламы ===
# Агрегация затрат по дате (если ещё не сделано)
ads_cost = df_ads.groupby('date')['cost'].sum().reset_index()

plt.figure(figsize=(14, 6), dpi=100)

# Линейный график с маркерами
plt.plot(ads_cost['date'], ads_cost['cost'], 
         color='#4A90E2', linewidth=1.5, marker='o', markersize=5)

# Подписи значений с чередованием (над/под точкой), чтобы не наезжали
for i, row in ads_cost.iterrows():
    offset = 12 if i % 2 == 0 else -25
    plt.text(row['date'], row['cost'] + offset, f"{int(row['cost'])} RUB", 
             ha='center', va='bottom' if offset > 0 else 'top', 
             fontsize=8, color='black', fontweight='bold')

# Оформление
plt.title('Aggregated Ad Campaign Costs (by day)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=11)
plt.ylabel('Cost (RUB)', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# Сохранение
plt.savefig('charts/ad_costs.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/ad_costs.png обновлён")

✅ charts/ad_costs.png обновлён


In [32]:
# === Визиты с выделением рекламных кампаний ===
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. Подготовка данных
visits_total = visits_by_date.copy()
visits_total['date'] = pd.to_datetime(visits_total['date'])
df_ads['date'] = pd.to_datetime(df_ads['date'])

# Вычисляем среднее количество визитов
avg_visits = visits_total['visits'].mean()

# 2. Построение графика
fig, ax = plt.subplots(figsize=(14, 6))

# Основная линия визитов
ax.plot(visits_total['date'], visits_total['visits'], 
        color='black', marker='o', markersize=5, linewidth=1.5, label='Visits')

# Горизонтальная линия среднего
ax.axhline(y=avg_visits, color='gray', linestyle='--', linewidth=1, label='Average Number of Visits')

# 3. Цветовые выделения для кампаний
# Группируем рекламу по названию кампании (и источнику/медиуму для точности)
# Используем уникальный набор (utm_source, utm_medium, utm_campaign)
campaign_groups = df_ads.groupby(['utm_source', 'utm_medium', 'utm_campaign'])

# Получаем набор цветов
import matplotlib.colors as mcolors
colors = list(mcolors.TABLEAU_COLORS.values()) 

for i, ((src, med, camp), group) in enumerate(campaign_groups):
    start_date = group['date'].min()
    end_date = group['date'].max()
    
    # Формируем подпись как в примере: "source medium campaign"
    # Для красоты можно заменить подчеркивания на пробелы, но в примере есть подчеркивания
    # Сделаем "google cpc virtual_reality_workshop"
    label_name = f"{src} {med} {camp}"
    
    # Цвет из списка
    color = colors[i % len(colors)]
    
    # Рисуем фон (span)
    ax.axvspan(start_date, end_date, color=color, alpha=0.4, label=label_name)

# 4. Оформление
ax.set_title("Visits during marketing active days", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Unique Visits", fontsize=12)
ax.set_xlabel("Date", fontsize=12)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, linestyle='--', alpha=0.3)
fig.autofmt_xdate(rotation=45)
plt.tight_layout()

# 5. Сохранение
plt.savefig('charts/visits_with_ads.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/visits_with_ads.png обновлён")

✅ charts/visits_with_ads.png обновлён


In [33]:
# === Регистрации с выделением рекламных кампаний ===
import matplotlib.colors as mcolors

# 1. Подготовка данных
regs_total = reg_by_date.copy()
regs_total['date'] = pd.to_datetime(regs_total['date'])
df_ads['date'] = pd.to_datetime(df_ads['date'])

# Вычисляем среднее количество регистраций
avg_regs = regs_total['registrations'].mean()

# 2. Построение графика
fig, ax = plt.subplots(figsize=(14, 6))

# Основная линия регистраций (Зеленая, как на скрине)
ax.plot(regs_total['date'], regs_total['registrations'], 
        color='#2E8B57', marker='o', markersize=5, linewidth=1.5, label='Registrations')

# Горизонтальная линия среднего (серая, пунктир)
ax.axhline(y=avg_regs, color='gray', linestyle='--', linewidth=1, label='Average Number of Registration')

# 3. Цветовые выделения для кампаний
campaign_groups = df_ads.groupby(['utm_source', 'utm_medium', 'utm_campaign'])
colors = list(mcolors.TABLEAU_COLORS.values()) 

for i, ((src, med, camp), group) in enumerate(campaign_groups):
    start_date = group['date'].min()
    end_date = group['date'].max()
    
    label_name = f"{src} {med} {camp}"
    color = colors[i % len(colors)]
    
    # Рисуем фон (прямоугольник)
    ax.axvspan(start_date, end_date, color=color, alpha=0.4, label=label_name)

# 4. Оформление
ax.set_title("Registrations during marketing active days", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Unique Users", fontsize=12)
ax.set_xlabel("Date", fontsize=12)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, linestyle='-', alpha=0.3) # Сетка тоньше
fig.autofmt_xdate(rotation=45)
plt.tight_layout()

# 5. Сохранение
plt.savefig('charts/registrations_with_ads.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ charts/registrations_with_ads.png обновлён")

✅ charts/registrations_with_ads.png обновлён


In [24]:
# Проверка: список созданных файлов
print("\n📁 Созданные графики:")
for f in sorted(os.listdir("charts")):
    print(f"  • {f}")


📁 Созданные графики:
  • ad_costs.png
  • average_conversion.png
  • conversion_by_platform.png
  • registrations_by_platform.png
  • registrations_with_ads.png
  • total_registrations.png
  • total_visits.png
  • visits_by_platform.png
  • visits_with_ads.png
